In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import random
import tensorflow as tf
import cv2

from src.font import FONT_PATHS, FONT_TO_LABEL, LABEL_TO_FONT_NAME

In [3]:
# Get the project directory
PROJECT_DIR = Path.cwd().parents[0]

In [4]:
# Sizes
IMG_SIZE = 64
MODEL_SIZE = 64
BATCH_SIZE = 32

In [5]:
# Digits from 1 to 9
DIGITS = list(range(1, 10))

In [6]:
# The paths of the fonts
NUM_CLASSES = len(FONT_TO_LABEL)

In [7]:
def render_digit(
    digit: int,
    font_path: str
):

    img = Image.new(
        "L",
        (IMG_SIZE, IMG_SIZE),
        255
    )

    draw = ImageDraw.Draw(img)

    font = scale_font_to_target(
        draw,
        font_path,
        str(digit),
        target_scale=random.uniform(
            0.65,
            0.9
        )
    )

    left, top, right, bottom = draw.textbbox(
        (0, 0),
        str(digit),
        font=font
    )

    w = right - left
    h = bottom - top

    x = (IMG_SIZE - w) // 2 + random.randint(-10, 10)
    y = (IMG_SIZE - h) // 2 + random.randint(-10, 10)

    draw.text(
        (x, y),
        str(digit),
        fill=0,
        font=font
    )

    angle = random.uniform(-10, 10)

    img = img.rotate(
        angle,
        fillcolor=255
    )

    return np.array(img)

In [8]:
def preprocess_digit(
    digit: np.ndarray,
    digit_target_size: int,
    canvas_size: int
) -> np.ndarray:

    h_digit, w_digit = digit.shape

    scale = digit_target_size / max(h_digit, w_digit)

    new_w = max(1, int(w_digit * scale))
    new_h = max(1, int(h_digit * scale))

    resized_digit = cv2.resize(
        digit,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros(
        (canvas_size, canvas_size),
        dtype=np.uint8
    )

    x_offset = (canvas_size - new_w) // 2
    y_offset = (canvas_size - new_h) // 2

    canvas[
        y_offset:y_offset + new_h,
        x_offset:x_offset + new_w
    ] = resized_digit

    return canvas.astype(np.float32) / 255.0


def scale_font_to_target(
    draw,
    font_path,
    text,
    target_scale=0.8
):
    target = IMG_SIZE * target_scale

    font = ImageFont.truetype(font_path, 20)

    for _ in range(10):

        left, top, right, bottom = draw.textbbox(
            (0, 0),
            text,
            font=font
        )

        w = right - left
        h = bottom - top

        scale = target / max(w, h)

        new_size = int(font.size * scale)

        if abs(scale - 1.0) < 0.05:
            break

        font = ImageFont.truetype(
            font_path,
            max(10, min(new_size, 300))
        )

    return font

In [9]:
def apply_camera_effects(
    arr: np.ndarray
):

    if random.random() < 0.7:

        sigma = random.uniform(
            0.3,
            2.0
        )

        arr = cv2.GaussianBlur(
            arr,
            (0, 0),
            sigma
        )

    if random.random() < 0.5:

        alpha = random.uniform(
            0.7,
            1.3
        )

        beta = random.uniform(
            -40,
            40
        )

        arr = cv2.convertScaleAbs(
            arr,
            alpha=alpha,
            beta=beta
        )

    if random.random() < 0.5:

        noise = np.random.normal(
            0,
            random.uniform(5, 20),
            arr.shape
        )

        arr = np.clip(
            arr + noise,
            0,
            255
        )

    if random.random() < 0.5:

        quality = random.randint(
            25,
            90
        )

        _, encoded = cv2.imencode(
            ".jpg",
            arr,
            [cv2.IMWRITE_JPEG_QUALITY, quality]
        )

        arr = cv2.imdecode(
            encoded,
            cv2.IMREAD_GRAYSCALE
        )

    return arr.astype(np.uint8)

In [10]:
def extract_digit_like_inference(
    img: np.ndarray
):

    blur = cv2.GaussianBlur(
        img,
        (3, 3),
        0
    )

    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    kernel = np.ones(
        (2, 2),
        np.uint8
    )

    thresh = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        kernel
    )

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    largest = max(
        contours,
        key=cv2.contourArea
    )

    area = cv2.contourArea(
        largest
    )

    if area < 20:
        return None

    x, y, w, h = cv2.boundingRect(
        largest
    )

    if w <= 0 or h <= 0:
        return None

    #
    # IMPORTANT:
    # Crop from grayscale image
    # not threshold image
    #
    digit_crop = blur[
        y:y+h,
        x:x+w
    ]

    digit_crop = 255 - digit_crop

    processed = preprocess_digit(
        digit_crop,
        int(MODEL_SIZE * 0.8),
        MODEL_SIZE
    )

    rgb = np.stack(
        [
            processed,
            processed,
            processed
        ],
        axis=-1
    )

    return rgb.astype(np.float32)

In [11]:
def generate_digit_sample():

    while True:

        digit = random.randint(1, 9)
        font_path = random.choice(FONT_PATHS)
        label = FONT_TO_LABEL[font_path]

        img = render_digit(digit, font_path)
        img = apply_camera_effects(img)

        # step 1: threshold like inference
        blur = cv2.GaussianBlur(img, (3, 3), 0)

        thresh = cv2.adaptiveThreshold(
            blur,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            11,
            2
        )

        kernel = np.ones((2, 2), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(
            thresh,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours:
            continue

        largest = max(contours, key=cv2.contourArea)

        if cv2.contourArea(largest) < 20:
            continue

        x, y, w, h = cv2.boundingRect(largest)

        digit_crop = thresh[y:y+h, x:x+w]

        processed = preprocess_digit(
            digit_crop,
            int(IMG_SIZE * 0.8),
            IMG_SIZE
        )

        sample = np.expand_dims(processed, axis=-1)

        return sample.astype(np.float32), label

In [12]:
def data_generator():
    """
    Data generator wrapper that yields an RGB image and its label

    Yields:
        tuple:
            np.ndarray: An RGB image that contains an augmented digit
            int: The label of the image (zero-based index)
    """

    while True:
        yield generate_digit_sample()

In [13]:
# fig, ax = plt.subplots(figsize=(6, 6))
# img, label= generate_digit_image(2)
# img = img.astype(np.uint8)
# ax.imshow(img)
# ax.set_title(f"Example of A Slice of Training Dataset ({LABEL_TO_FONT_NAME[label]})", pad=2)
# ax.axis('off')

# plt.tight_layout(pad=1.0)
# plt.show()

In [14]:
# Describe what the custom generator will yield
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32), # the image
    tf.TensorSpec(shape=(), dtype=tf.int32) # the label
)

# Create the training dataset from the generator
train_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

# Use prefetching to improve performance
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

2026-06-07 15:42:17.414532: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-07 15:42:17.414551: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-06-07 15:42:17.414554: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1780864937.414571 1055586 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1780864937.414591 1055586 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [15]:
# Create the training dataset from the generator
val_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [16]:
# Create a model
model = tf.keras.Sequential([

    # First layer
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(MODEL_SIZE, MODEL_SIZE, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Second layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Third layer
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Forth layer
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Final dense layer for classification
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary of the model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 4, 4, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 2, 2, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 697,797 (2.66 MB)

 Trainable params: 696,837 (2.66 MB)

 Non-trainable params: 960 (3.75 KB)

In [18]:
x, y = next(iter(train_ds))
print(x.shape)

(32, 64, 64, 1)


[ WARN:0@0.984] global loadsave.cpp:1671 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


In [19]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [20]:
history = model.fit(
    train_ds,
    steps_per_epoch=200,
    epochs=20,
    callbacks=[callback]
)

Epoch 1/20


2026-06-07 15:42:18.271776: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.2917 - loss: 6.3771
Epoch 2/20
  5/200 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.2711 - loss: 6.2415

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.3459 - loss: 6.5768
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.3914 - loss: 6.6131
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.4470 - loss: 6.3159
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.5642 - loss: 5.5625
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6475 - loss: 4.5101
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6839 - loss: 3.9942
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.7220 - loss: 3.3108
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.7411 - loss: 3.0313
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.7748 - loss: 2.5698
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.8075 - loss: 2.1269
Epoch 12/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.8261 - loss: 1.8326
Epoch 13/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/st

In [21]:
# Evaluate the model
test_loss, test_acc = model.evaluate(val_ds, steps=5, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

5/5 - 0s - 49ms/step - accuracy: 0.9500 - loss: 0.3879

Test Accuracy: 95.00%


In [22]:
# Save the model and its parameters
model.save(f'{PROJECT_DIR}/models/font_recognition_MobileNetV2.keras')